# Block 3 — Time-Series Emergency Insurance

**Target: 30–45 minutes. Real data: UCI Bike Sharing.**

In [1]:
import sys
from pathlib import Path
import numpy as np,pandas as pd
from sklearn.metrics import mean_squared_error
ROOT=next((p for p in [Path.cwd()]+list(Path.cwd().parents) if (p/"reference"/"dataset_loader.py").exists()),Path.cwd())
sys.path.insert(0,str(ROOT/"reference"))
from dataset_loader import load_checked_csv
URL="https://raw.githubusercontent.com/KeithJLZ/UCI-Bike-Sharing-Dataset/master/hour.csv"
df=load_checked_csv("bike_hour.csv",URL,["dteday","hr","temp","hum","cnt"],17000); df["dteday"]=pd.to_datetime(df["dteday"]); df=df.sort_values(["dteday","hr"]).reset_index(drop=True)

## Memorise 5 rules
1. sort by time
2. do not random-shuffle
3. target, lag target/features
4. rolling features use past information only!
5. persistence / seasonal-naive baseline before ML

Create full timestamp and next-hour target.

In [13]:
df['timestamp'] = df['dteday'] + pd.to_timedelta(df['hr'], unit='h')
df['target'] = df['cnt'].shift(-1).astype('Int64')
display(df.head())

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt,timestamp,target
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16,2011-01-01 00:00:00,40
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40,2011-01-01 01:00:00,32
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32,2011-01-01 02:00:00,13
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13,2011-01-01 03:00:00,1
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1,2011-01-01 04:00:00,1


Create lag1, lag24, rolling24 mean with correct shift.

In [14]:
df['lag1'] = df['cnt'].shift(1)
df['lag24'] = df['cnt'].shift(24)

df['rolling24'] = df['cnt'].shift(1).rolling(24).mean()

display(df.head())

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,...,hum,windspeed,casual,registered,cnt,timestamp,target,lag1,lag24,rolling24
0,1,2011-01-01,1,0,1,0,0,6,0,1,...,0.81,0.0,3,13,16,2011-01-01 00:00:00,40,NaN,NaN,NaN
1,2,2011-01-01,1,0,1,1,0,6,0,1,...,0.80,0.0,8,32,40,2011-01-01 01:00:00,32,16.0,NaN,NaN
2,3,2011-01-01,1,0,1,2,0,6,0,1,...,0.80,0.0,5,27,32,2011-01-01 02:00:00,13,40.0,NaN,NaN
3,4,2011-01-01,1,0,1,3,0,6,0,1,...,0.75,0.0,3,10,13,2011-01-01 03:00:00,1,32.0,NaN,NaN
4,5,2011-01-01,1,0,1,4,0,6,0,1,...,0.75,0.0,0,1,1,2011-01-01 04:00:00,1,13.0,NaN,NaN


Create hour sine/cosine.

In [ ]:
df['hour_sin'] = np.sin(2 * np.pi * df['hr']/24)
df['hour_cos'] = np.cos(2 * np.pi * df['hr']/24)

Chronological 80/20 split and persistence / 24h seasonal-naive RMSE.

In [19]:
df['lag7'] = df['cnt'].shift(7)
df['ordinal'] = np.arange(len(df.index))

new_df = df.dropna().sort_values('timestamp').copy()
split = int(0.8 * len(new_df.index))  # chron 80/20 split
train, valid = new_df.iloc[:split], new_df.iloc[split:]

persistence = valid['lag1']  # daily_naive
weekly_naive = valid['lag7']

from sklearn.metrics import mean_absolute_error
print(mean_absolute_error(valid['target'], persistence))
print(mean_absolute_error(valid['target'], weekly_naive))


145.12791702679343
278.8985883030827


In [33]:
# Naive time-target LR fit
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

trend = LinearRegression()
trend.fit(train[['ordinal']], train['cnt'])
trend_pred = trend.predict(valid[['ordinal']])

train_residual = train['cnt'] - trend.predict(train[['ordinal']])
valid_residual = valid['cnt'] - trend_pred

print(root_mean_squared_error(valid['cnt'], trend_pred), mean_absolute_error(valid['cnt'], trend_pred))
print(valid_residual.abs().mean())



226.3393768460993 187.436249820848
187.436249820848


In [36]:
# Hybrid model: level (trend) + departure
from sklearn.ensemble import RandomForestRegressor

feature_cols = ['ordinal', 'lag1', 'lag7', 'lag24', 'rolling24']

rf = RandomForestRegressor(random_state=42).fit(train[feature_cols], train_residual)
final_pred = trend.predict(valid[['ordinal']]) + rf.predict(valid[feature_cols])

print(root_mean_squared_error(valid['cnt'], final_pred), mean_absolute_error(valid['cnt'], final_pred))


92.09932788495492 66.49545795291093


## Answer pattern
```python
df["timestamp"]=df["dteday"]+pd.to_timedelta(df["hr"],unit="h")
df["target"]=df["cnt"].shift(-1)
df["lag1"]=df["cnt"].shift(1)
df["lag24"]=df["cnt"].shift(24)
df["roll24"]=df["cnt"].shift(1).rolling(24).mean()
df["hour_sin"]=np.sin(2*np.pi*df["hr"]/24)
df["hour_cos"]=np.cos(2*np.pi*df["hr"]/24)
```